# 整数×連続の厳密線形化 — 適用前・原理・適用・効果

「整数個 × 連続量」(バッチ数×バッチサイズ、台数×稼働時間 など)の積が制約に入ると、
SCIP は一般の双線形項として **McCormick 緩和**で扱う。整数側の情報が緩和で捨てられるため
下界が緩み、gap がなかなか縮まない。

この notebook は次の一貫した型でこの現象と対処を追う。

1. **課題(before)** — 双線形のまま `mk.analyze` で診断し `weak_relaxation` を確認する
2. **原理(principle)** — McCormick 包絡線のギャップを図で見る
3. **適用(how)** — `mk.linearize_product` を1行当てる
4. **効果(before/after)** — `mk.compare_variants` でルート双対境界・gap・ノード数を比較する

題材は **バッチ反応器スケジューリング**(`samples/others/scheduling_plant.py`)。
バッチ数 `n`(整数)× バッチサイズ `s`(連続)の積が需要充足とエネルギーの三重積に入る。

In [1]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" 有無では docs で止まる。pyproject.toml を目印にする。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import scheduling_plant as sp

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 課題(before) — baseline を診断する

双線形のまま(`linearize_ns=False`)のモデルを `mk.analyze` に掛ける。
`analyze` は実際に解いて双対境界の停滞・空間分枝の偏り・非線形制約の違反を収集し、
症状を重要度順に返す。

In [2]:
report = mk.analyze(lambda: sp.build_model(linearize_ns=False),
                    name="plant-baseline", time_limit=10)
print(report.summary())
print("\n支配ボトルネック:", report.metrics.get("bottleneck_type"),
      "/ 空間分枝の寄与:", f"{report.metrics.get('spatial_share', 0):.1%}")

[plant-baseline] 検出症状 4件:
  [serious] 特定の非線形制約に緩和違反が集中(かつ空間分枝が多い) -> ボトルネック制約の区分線形近似・凸包再定式化・変数境界タイト化
  [warning] 双対境界の改善が停滞(gapが残る) -> 有効不等式の追加・変数境界タイト化・Big-M排除で緩和強化
  [warning] 係数の絶対値レンジが桁違い / Big-M候補あり(presolve後も残存) -> スケーリング、Big-MのIndicator/SOS制約化、係数の再定式化
  [good] 制約-変数がブロック構造 + 少数の結合制約 -> ベンダーズ分解 / Dantzig-Wolfe分解(結合制約を主問題に)

支配ボトルネック: energy / 空間分枝の寄与: 55.0%


三重積 $n\cdot s\cdot(T-T_0)$(energy 制約)などがボトルネックとして挙がり、
`weak_relaxation`(緩和が弱い)が発火する。recipe は「整数×連続は `mk.linearize_product`」。
次でその根拠を図で確認する。

## 2. 原理(principle) — McCormick 包絡線のギャップ

整数 $n$ と連続 $s$ の積 $w = ns$ を考える。SCIP は $n$ を **連続緩和**して
双線形項に McCormick 包絡線を当てる。すると $(s, w)$ 平面では
$n=1$ の直線 $w=s$ から $n=3$ の直線 $w=3s$ までの帯状領域**すべて**が緩和で許容される。
しかし整数の真の実行可能集合は $w=1\cdot s,\ 2\cdot s,\ 3\cdot s$ の**3本の直線だけ**。
帯の内部(下図の網掛け)が緩和のギャップ = 下界を緩めている余地である。

In [3]:
# n in {1,2,3}, s in [0,10] で w=n*s を描く(小さい値域で図を見やすく)
s = np.linspace(0, 10, 100)
fig = go.Figure(layout=base_layout(
    "整数×連続の積 w = n·s : 緩和が許す領域 vs 真の実行可能直線",
    "連続量 s", "積 w = n·s", height=420))
# McCormick(n を連続緩和)が許す帯: s <= w <= 3s
fig.add_trace(go.Scatter(x=np.r_[s, s[::-1]], y=np.r_[3*s, s[::-1]],
    fill="toself", fillcolor="rgba(194,90,0,0.12)", line=dict(width=0),
    name="緩和が許す領域 (n を連続扱い)", hoverinfo="skip"))
# 真の整数直線 n=1,2,3
for v, col in zip([1, 2, 3], [C["s1"], C["s2"], C["ink"]]):
    fig.add_trace(go.Scatter(x=s, y=v*s, mode="lines",
        line=dict(color=col, width=2.5), name=f"n={v} (真の値 w={v}·s)",
        hovertemplate="n=" + str(v) + ": w=%{y:.1f}<extra></extra>"))
# ギャップの可視化: s=8 での縦幅
fig.add_annotation(x=8, y=(1*8+3*8)/2, ax=8, ay=1*8,
    text="緩和ギャップ", showarrow=True, arrowhead=2, arrowcolor=C["warn"],
    font=dict(color=C["warn"], size=12), xshift=40)
fig.add_shape(type="line", x0=8, y0=1*8, x1=8, y1=3*8,
    line=dict(color=C["warn"], width=1.5, dash="dot"))
show(fig)

**厳密線形化はこの帯を消す。** $n$ の各値 $v$ に指示変数 $\delta_v$($n=v$ で1)を作り、
$s$ を $s = \sum_v s_v$($s_v$ は $\delta_v=1$ のときだけ有効)に分解して
$w = \sum_v v\,s_v$ と書くと、緩和の実行可能集合が **真の3本の直線の凸包**に一致し、
McCormick の帯のような「整数の隙間を埋める」ギャップが原理的に消える(緩和ギャップ0の厳密変換)。

## 3. 適用(how) — `mk.linearize_product`

汎用ヘルパーを1行呼ぶだけで積を厳密線形化できる。`scheduling_plant.build_model(linearize_ns=True)`
は各ジョブの `n·s` をこの1行に置き換えている。

```python
ns = mk.linearize_product(m, n, s, y_lb=1, y_ub=N_MAX, x_lb=S_MIN, x_ub=S_MAX, name="ns")
# 以降 ns は n*s の厳密線形表現。ns*X や ns*(T-T0) の双線形に落ちる
```

下は最小の動作確認。`n·s ≥ 12` を厳密線形制約として解く。

In [4]:
from pyscipopt import Model
m = Model(); m.hideOutput()
n = m.addVar(vtype="I", lb=1, ub=3, name="n")
s = m.addVar(lb=0.0, ub=10.0, name="s")
ns = mk.linearize_product(m, n, s, 1, 3, 0.0, 10.0, "ns")
m.addCons(ns >= 12)
m.setObjective(n + s, "minimize")
m.optimize()
print(f"n={m.getVal(n):.0f}  s={m.getVal(s):.2f}  n*s={m.getVal(ns):.2f}  (>= 12 を厳密に満たす)")

n=3  s=4.00  n*s=12.00  (>= 12 を厳密に満たす)


## 4. 効果(before/after) — `mk.compare_variants`

双線形のまま(baseline)と厳密線形化(linearized)を同じ時間制限で解き、
**ルート双対境界・最終gap・ノード数**を比較する。定式化の質は探索動学に交絡されない
**ルート双対境界**で測るのが本筋(gap/ノードは参考)。

In [5]:
df = mk.compare_variants(
    {"baseline(bilinear)": lambda: sp.build_model(linearize_ns=False),
     "linearized":         lambda: sp.build_model(linearize_ns=True)},
    time_limit=10)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

,variant,root_dual,final_dual,final_gap,nodes
0,baseline(bilinear),52.128421,90.495483,1.552767,2580
1,linearized,133.302921,143.835957,0.605664,848


In [6]:
base = df.loc[df["variant"] == "baseline(bilinear)"].iloc[0]
lin  = df.loc[df["variant"] == "linearized"].iloc[0]

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (高いほど良い)", "最終 gap [%] (低いほど良い)",
                    "探索ノード数 (少ないほど良い)"))
labels = ["baseline", "linearized"]
bar_colors = [C["muted"], C["s1"]]
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [base["root_dual"], lin["root_dual"]], lambda v: f"{v:.0f}")
add_bars(2, [base["final_gap"]*100, lin["final_gap"]*100], lambda v: f"{v:.0f}%")
add_bars(3, [base["nodes"], lin["nodes"]], lambda v: f"{int(v)}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="厳密線形化の before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)

In [7]:
print(f"ルート双対境界 : {base['root_dual']:.1f} -> {lin['root_dual']:.1f} "
      f"(+{(lin['root_dual']/base['root_dual']-1)*100:.0f}%)")
print(f"最終gap        : {base['final_gap']:.1%} -> {lin['final_gap']:.1%}")
print(f"ノード数       : {int(base['nodes'])} -> {int(lin['nodes'])}")

ルート双対境界 : 52.1 -> 133.3 (+156%)
最終gap        : 155.3% -> 60.6%
ノード数       : 2580 -> 848


## まとめ

- **ルート双対境界が大きく上がる**。McCormick 帯のギャップが消え、最小化問題の下界が
  最初から締まる。締まった下界のぶん最終gapもノード数も縮む。
- 最適値そのものは変わらない(等価変換)。**緩和だけを締める**改善である。

### なぜ SCIP が自動でやらないのか

SCIP は双線形項に汎用の McCormick 緩和を当てる。「整数側を指示変数で分解すれば厳密になる」
という**問題構造に依存した再定式化**は、build 済みモデルからは自動検出できない
(SCIP presolve の範囲外)。だから診断が構造を指摘し、モデラーが `mk.linearize_product` を
1行当てる、という分業になる。

### 効果は問題依存

- **整数側の値域が広い**と補助変数・制約が線形に増える。バッチ数×バッチサイズのように
  整数側が小さい積に向く。
- 効くのは「非凸緩和の弱さが律速」のケース限定。[診断ベンチマーク](../../census.md)では
  非線形11本中 `weak_relaxation` 発火は1本のみ。だからこそ `mk.analyze` で確認し、
  `mk.compare_variants` で**効いたことを測ってから**採用する手順が要になる。

関連: [手法ガイド 1. 整数×連続の厳密線形化](../../playbook/01-linearize.md) /
API [`mk.linearize_product`](../../api/transforms.md)